In [1]:
import pandas as pd
import json
import random

df = pd.read_parquet('./human-00000-of-00001-25f4910818759289.parquet')
df_machine = pd.read_parquet('./gpt4_pair-00000-of-00001-c0b431264a82ddc0.parquet')

In [2]:
print(df[0:1])
print(df_machine[0:1])

   question_id     model_a        model_b   winner     judge  \
0           81  alpaca-13b  gpt-3.5-turbo  model_b  author_2   

                                      conversation_a  \
0  [{'content': 'Compose an engaging travel blog ...   

                                      conversation_b  turn  
0  [{'content': 'Compose an engaging travel blog ...     1  
   question_id     model_a    model_b   winner      judge  \
0           81  alpaca-13b  claude-v1  model_b  gpt4_pair   

                                      conversation_a  \
0  [{'content': 'Compose an engaging travel blog ...   

                                      conversation_b  turn  
0  [{'content': 'Compose an engaging travel blog ...     1  


In [3]:
print(df.columns)
print(df_machine.columns)
print(len(df))
print(len(df_machine))

Index(['question_id', 'model_a', 'model_b', 'winner', 'judge',
       'conversation_a', 'conversation_b', 'turn'],
      dtype='object')
Index(['question_id', 'model_a', 'model_b', 'winner', 'judge',
       'conversation_a', 'conversation_b', 'turn'],
      dtype='object')
3355
2400


In [42]:
threshold = 600
low = 200
results = []
len12, len34 = 0,0
print(len(df))
nexts = ["\n","1.","[","```","earlier reply"]
judge_list = []

for i in range(len(df)):
    len_1 = len(df['conversation_a'][i][1]['content'])
    len_3 = len(df['conversation_a'][i][3]['content'])
    len_2 = len(df['conversation_b'][i][1]['content'])
    len_4 = len(df['conversation_b'][i][3]['content'])
    if len_1 < threshold and len_1 > low and not any(next in df['conversation_a'][i][1]['content'] for next in nexts):
        if len_2 < threshold and len_2 > low and not any(next in df['conversation_b'][i][1]['content'] for next in nexts):
            if not any (next in df['conversation_a'][i][0]['content'] for next in nexts):
                len12 += 1
                dict_all = {'question_id': (int)(df['question_id'][i]),
                    'model_a': df['model_a'][i],
                    'model_b': df['model_b'][i],
                    'question': df['conversation_a'][i][0]['content'],
                    'conversation_a': df['conversation_a'][i][1]['content'],
                    'conversation_b': df['conversation_b'][i][1]['content'],
                }
                if dict_all not in results:
                    if df['winner'][i] == 'model_a':
                        score = 1
                    elif df['winner'][i] == 'model_b':
                        score = 2
                    elif df['winner'][i] == 'tie':
                        score = 1.5
                    judges = {'question_id': (int)(df['question_id'][i]),
                              'conversation_a': df['conversation_a'][i][1]['content'],
                              'conversation_b': df['conversation_b'][i][1]['content'],
                              'model_a': df['model_a'][i],
                              'model_b': df['model_b'][i],
                              'winner': df['winner'][i],
                              'score': score,
                              'judge': df['judge'][i]}
                    judge_list.append(judges)
                    results.append(dict_all)
    if len_3 < threshold and len_3 > low and not any(next in df['conversation_a'][i][3]['content'] for next in nexts):
        if len_4 < threshold and len_4 > low and not any(next in df['conversation_b'][i][3]['content'] for next in nexts):
            if not any(next in df['conversation_a'][i][2]['content'] for next in nexts):
                len34 += 1
                dict_all = {'question_id': (int)(df['question_id'][i]),
                    'model_a': df['model_a'][i],
                    'model_b': df['model_b'][i],
                    'question': df['conversation_a'][i][2]['content'],
                    'conversation_a': df['conversation_a'][i][3]['content'],
                    'conversation_b': df['conversation_b'][i][3]['content'],
                }
                if dict_all not in results:
                    if df['winner'][i] == 'model_a':
                        score = 1
                    elif df['winner'][i] == 'model_b':
                        score = 2
                    elif df['winner'][i] == 'tie':
                        score = 1.5
                    judges = {'question_id': (int)(df['question_id'][i]),
                              'conversation_a': df['conversation_a'][i][3]['content'],
                              'conversation_b': df['conversation_b'][i][3]['content'],
                              'model_a': df['model_a'][i],
                              'model_b': df['model_b'][i],
                              'winner': df['winner'][i],
                              'score': score,
                              'judge': df['judge'][i]}
                    judge_list.append(judges)
                    results.append(dict_all)
print(len12,len34)
print(len(results))
random.shuffle(results)

3355
115 49
53


In [49]:
new_judge_list = []
for item in judge_list:
    for item2 in df_machine.to_dict('records'):
        if item['question_id'] == item2['question_id'] and item['model_a'] == item2['model_a'] and item['model_b'] == item2['model_b']:
            if item2['winner'] == 'model_a':
                score = 1
            elif item2['winner'] == 'model_b':
                score = 2
            elif item2['winner'] == 'tie':
                score = 1.5
            judge_list_add_machine = {'question_id': item['question_id'],
                                      'conversation_a': item['conversation_a'],
                                      'conversation_b': item['conversation_b'],
                                      'model_a': item['model_a'],
                                      'model_b': item['model_b'],
                                      'winner': item['winner'],
                                      'score': item['score'],
                                      'gpt4_score': score,
                                      'judge': item['judge']}
            new_judge_list.append(judge_list_add_machine)
            break
        elif item['question_id'] == item2['question_id'] and item['model_a'] == item2['model_b'] and item['model_b'] == item2['model_a']:
            if item2['winner'] == 'model_a':
                score = 2
            elif item2['winner'] == 'model_b':
                score = 1
            elif item2['winner'] == 'tie':
                score = 1.5
            judge_list_add_machine = {'question_id': item['question_id'],
                                      'conversation_a': item['conversation_a'],
                                      'conversation_b': item['conversation_b'],
                                      'model_a': item['model_a'],
                                      'model_b': item['model_b'],
                                      'winner': item['winner'],
                                      'score': item['score'],
                                      'gpt4_score': score,
                                      'judge': item['judge']}
            new_judge_list.append(judge_list_add_machine)
            break
                

In [50]:
print(len(judge_list))
print(len(new_judge_list))

53
53


In [51]:
judge_list = pd.DataFrame(judge_list)
judge_list.to_csv('mt-bench.csv',index=False)
new_judge_list = pd.DataFrame(new_judge_list)
new_judge_list.to_csv('mt-bench-gpt4.csv',index=False)

In [6]:
filtered_df = pd.DataFrame(results)
filtered_df.to_parquet('filtered_conversations.parquet', index=False)
df = pd.read_parquet('./filtered_conversations.parquet')
for i in range(10):
    print(df['question_id'][i])
    print(df['model_a'][i],df['model_b'][i])

92
alpaca-13b gpt-3.5-turbo
92
gpt-3.5-turbo vicuna-13b-v1.2
92
gpt-3.5-turbo vicuna-13b-v1.2
98
llama-13b claude-v1
92
vicuna-13b-v1.2 gpt-3.5-turbo
92
vicuna-13b-v1.2 gpt-3.5-turbo
100
llama-13b alpaca-13b
98
llama-13b gpt-3.5-turbo
98
claude-v1 llama-13b
92
claude-v1 vicuna-13b-v1.2


In [2]:
import pandas as pd
df = pd.read_parquet('./filtered_conversations.parquet')
max_len = 0
ques_ids = set()
models = set()
for i in range(len(df)):
    ques_ids.add(df['question_id'][i])
    models.add((df['model_a'][i],df['model_b'][i]))
    models.add((df['model_b'][i],df['model_a'][i]))
    if len(df['conversation_a'][i])>max_len:
        max_len = len(df['conversation_a'][i])
        max_con = df['conversation_a'][i]
    if len(df['conversation_b'][i])>max_len:
        max_len = len(df['conversation_b'][i])
        max_con = df['conversation_b'][i]
    # if len(df['question'][i])>max_len:
    #     max_len = len(df['question'][i])
    #     max_con = df['question'][i]
print(max_len)
print(max_con)
print(len(ques_ids))
print(models)
print(len(models))

568
The bus? With all those strangers and their germs? I think not. I'll drive us in my spotless and germ-free vehicle. *pretends to grab car keys* To the Cheesecake Factory! *makes driving sound effects* We have arrived. I've already looked at the menu and nutrition information online so I know exactly what to order to maximize taste and minimize unhealthy fats. May I suggest we start with an appetizer of lettuce wraps and an entree of pasta primavera with grilled chicken? I calculated that should provide optimal nutrition as well as leftovers for tomorrow's lunch.
11
{('vicuna-13b-v1.2', 'claude-v1'), ('gpt-3.5-turbo', 'claude-v1'), ('vicuna-13b-v1.2', 'alpaca-13b'), ('gpt-3.5-turbo', 'alpaca-13b'), ('alpaca-13b', 'llama-13b'), ('gpt-4', 'claude-v1'), ('claude-v1', 'vicuna-13b-v1.2'), ('gpt-4', 'alpaca-13b'), ('vicuna-13b-v1.2', 'gpt-3.5-turbo'), ('vicuna-13b-v1.2', 'gpt-4'), ('gpt-3.5-turbo', 'gpt-4'), ('alpaca-13b', 'claude-v1'), ('llama-13b', 'claude-v1'), ('llama-13b', 'alpaca-13

In [3]:
import pandas as pd

df = pd.read_parquet('./gpt4_pair-00000-of-00001-c0b431264a82ddc0.parquet')
print(df.head())
print(df.columns)
print(df.shape)
print(df.dtypes)

   question_id     model_a        model_b   winner      judge  \
0           81  alpaca-13b      claude-v1  model_b  gpt4_pair   
1           81  alpaca-13b      claude-v1  model_b  gpt4_pair   
2           81  alpaca-13b  gpt-3.5-turbo  model_b  gpt4_pair   
3           81  alpaca-13b  gpt-3.5-turbo  model_b  gpt4_pair   
4           81  alpaca-13b          gpt-4  model_b  gpt4_pair   

                                      conversation_a  \
0  [{'content': 'Compose an engaging travel blog ...   
1  [{'content': 'Compose an engaging travel blog ...   
2  [{'content': 'Compose an engaging travel blog ...   
3  [{'content': 'Compose an engaging travel blog ...   
4  [{'content': 'Compose an engaging travel blog ...   

                                      conversation_b  turn  
0  [{'content': 'Compose an engaging travel blog ...     1  
1  [{'content': 'Compose an engaging travel blog ...     2  
2  [{'content': 'Compose an engaging travel blog ...     1  
3  [{'content': 'Compose an 

In [4]:
print(df['question_id'][0])
print(df['model_a'][0])
print(df['model_b'][0])
print(df['winner'][0])
print(df['judge'][0])
print(df['conversation_a'][0])
print(df['conversation_b'][0])
print(df['turn'][0])

81
alpaca-13b
claude-v1
model_b
gpt4_pair
[{'content': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.', 'role': 'user'}
 {'content': 'I recently had the pleasure of visiting Hawaii and it quickly became one of my favorite places. From the stunning beaches to the lush mountains, this place has it all. The people are incredibly friendly and the culture is alive and well. One of the highlights of my trip was visiting the Polynesian Cultural Center. Here, I was able to learn about the culture of the native Hawaiian people and try my hand at traditional crafts and activities. I also had a chance to explore some of the natural wonders of the island, including the breathtaking Hanauma Bay and the majestic Waimea Canyon. Whether you’re looking for a relaxing beach vacation or an adventure filled with culture and nature, Hawaii is the perfect destination.', 'role': 'assistant'}
 {'content': 'Rewrite your previous 

In [5]:
import pandas as pd

df = pd.read_parquet('./human-00000-of-00001-25f4910818759289.parquet')
print(df.head())
print(df.columns)
print(df.shape)
print(df.dtypes)

   question_id     model_a          model_b   winner      judge  \
0           81  alpaca-13b    gpt-3.5-turbo  model_b   author_2   
1           81  alpaca-13b    gpt-3.5-turbo  model_b   author_2   
2           81  alpaca-13b    gpt-3.5-turbo  model_b  expert_17   
3           81  alpaca-13b    gpt-3.5-turbo  model_b  expert_17   
4           81  alpaca-13b  vicuna-13b-v1.2  model_b   expert_0   

                                      conversation_a  \
0  [{'content': 'Compose an engaging travel blog ...   
1  [{'content': 'Compose an engaging travel blog ...   
2  [{'content': 'Compose an engaging travel blog ...   
3  [{'content': 'Compose an engaging travel blog ...   
4  [{'content': 'Compose an engaging travel blog ...   

                                      conversation_b  turn  
0  [{'content': 'Compose an engaging travel blog ...     1  
1  [{'content': 'Compose an engaging travel blog ...     2  
2  [{'content': 'Compose an engaging travel blog ...     1  
3  [{'content': 

In [6]:
print(df['question_id'][0])
print(df['model_a'][0])
print(df['model_b'][0])
print(df['winner'][0])
print(df['judge'][0])
print(df['conversation_a'][0])
print(df['conversation_b'][0])
print(df['turn'][0])

81
alpaca-13b
gpt-3.5-turbo
model_b
author_2
[{'content': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.', 'role': 'user'}
 {'content': 'I recently had the pleasure of visiting Hawaii and it quickly became one of my favorite places. From the stunning beaches to the lush mountains, this place has it all. The people are incredibly friendly and the culture is alive and well. One of the highlights of my trip was visiting the Polynesian Cultural Center. Here, I was able to learn about the culture of the native Hawaiian people and try my hand at traditional crafts and activities. I also had a chance to explore some of the natural wonders of the island, including the breathtaking Hanauma Bay and the majestic Waimea Canyon. Whether you’re looking for a relaxing beach vacation or an adventure filled with culture and nature, Hawaii is the perfect destination.', 'role': 'assistant'}
 {'content': 'Rewrite your previo

In [ ]:
import pandas as pd
import json

df = pd.read_parquet('./human-00000-of-00001-25f4910818759289.parquet')
threshold = 1500
low = 1000
results = []
a_len, b_len,ab_len = 0, 0,0
print(len(df))

for i in range(len(df)):
    len_a = len(df['conversation_a'][i][0]['content'])+len(df['conversation_a'][i][1]['content'])+len(df['conversation_a'][i][2]['content'])+len(df['conversation_a'][i][3]['content'])
    len_b = len(df['conversation_b'][i][0]['content'])+len(df['conversation_b'][i][1]['content'])+len(df['conversation_b'][i][2]['content'])+len(df['conversation_b'][i][3]['content'])
    if len_a < threshold and len_a > low:
        a_len += 1
        if len_b < threshold and len_b > low:
            ab_len += 1
            # print(i,len_a,df['conversation_a'][i])
            # print(i,len_b,df['conversation_b'][i])
            results.append({
                'question_id': i,
                'conversation_a': df['conversation_a'][i],
                'conversation_b': df['conversation_b'][i]
            })
print(a_len,ab_len)
print(a_len/len(df), ab_len/len(df))

3355
149 1044 [{'content': 'Describe a vivid and unique character, using strong imagery and creative language. Please answer in fewer than two paragraphs.', 'role': 'user'}
 {'content': "He was a giant of a man, with a gruff exterior and a heart of gold. His eyes were the color of the sea, and his hands were as big and powerful as a lion's paws. He had a beard that was as wild and untamed as the wind, and his voice was like a thunderclap echoing through the mountains. He was a man of few words, but when he spoke, his words carried weight and wisdom.", 'role': 'assistant'}
 {'content': 'Revise your previous response and incorporate an allusion to a famous work of literature or historical event in each sentence.', 'role': 'user'}
 {'content': "He was a giant of a man, with a gruff exterior like the mighty oak tree from The Lord of the Rings; his eyes were as deep and mysterious as the ocean in Homer's Odyssey; his hands were as big and powerful as a lion's paws from the Bible; his beard 

In [8]:
filtered_df = pd.DataFrame(results)
filtered_df.to_parquet('filtered_conversations.parquet', index=False)